In [ ]:
from thesis_utils import APP_ROOT, DATA_ROOT, PREVIEW_ROWS, PROJECT_ROOT, paths
# ============================================================
# S&P 500 HEADLINES -> FINBERT LABELING
#
# DÜZELTİLMİŞ SÜRÜM
#
# Amaç:
# - D:\serkan.kaymak\financial_sentiment_thesis\db içinden S&P 500 headline dosyasını oku
# - headline/text kolonunu otomatik bul
# - FinBERT ile label + confidence + probability kolonları üret
# - Orijinal dosyayı bozmadan *_labeled olarak kaydet
# - Yarıda kesilirse *_labeled_progress.parquet üzerinden devam et
#
# Çıktılar:
# - <orijinal_dosya_adi>_labeled.parquet
# - <orijinal_dosya_adi>_labeled.csv
# - <orijinal_dosya_adi>_labeled.xlsx
# - <orijinal_dosya_adi>_labeled_progress.parquet
# ============================================================

from pathlib import Path
import sys
import subprocess
import random
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 0) Paket kontrol
# ------------------------------------------------------------
# ------------------------------------------------------------
# 0) Paket kontrol
# ------------------------------------------------------------
def pip_install(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        __import__(import_name)
    except Exception:
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            package_name
        ])

pip_install("torch")
pip_install("transformers")
pip_install("tqdm")
pip_install("openpyxl")
pip_install("pyarrow")

import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ------------------------------------------------------------
# 1) Ayarlar
# ------------------------------------------------------------
DATA_DIR = DATA_ROOT

MODEL_NAME = "ProsusAI/finbert"
MAX_LENGTH = 128
BATCH_SIZE = 32
RANDOM_STATE = 42
SAVE_EVERY_BATCHES = 25

# Eğer otomatik yanlış dosyayı seçerse buraya tam dosya yolunu yaz:
# FILE_PATH_OVERRIDE = Path(r"D:\serkan.kaymak\financial_sentiment_thesis\db\dosyanin_tam_adi.xlsx")
FILE_PATH_OVERRIDE = paths.SP500_RAW_HEADLINES_PATH

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("DATA_DIR exists:", DATA_DIR.exists())
print("DATA_DIR:", DATA_DIR)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Klasör bulunamadı: {DATA_DIR}")

# ------------------------------------------------------------
# 2) Dosyayı bul
# ------------------------------------------------------------
allowed_exts = [".xlsx", ".xls", ".csv", ".parquet", ".json", ".jsonl"]

if FILE_PATH_OVERRIDE is not None:
    FILE_PATH = Path(FILE_PATH_OVERRIDE)
    if not FILE_PATH.exists():
        raise FileNotFoundError(f"Elle verilen dosya bulunamadı: {FILE_PATH}")
else:
    all_files = [
        p for p in DATA_DIR.iterdir()
        if p.is_file()
        and p.suffix.lower() in allowed_exts
        and "_labeled" not in p.stem.lower()
        and "_progress" not in p.stem.lower()
    ]

    print("\nKlasördeki uygun dosyalar:")
    for i, p in enumerate(all_files):
        print(i, "->", p.name)

    keywords = [
        "sp500",
        "s&p",
        "s_p",
        "s and p",
        "headline",
        "headlines",
        "financial",
        "news",
        "2008",
        "2024",
    ]

    candidate_files = []

    for p in all_files:
        name = p.name.lower()
        score = sum(1 for kw in keywords if kw in name)
        if score >= 1:
            candidate_files.append((score, p))

    candidate_files = sorted(candidate_files, key=lambda x: x[0], reverse=True)

    if not candidate_files:
        raise FileNotFoundError(
            "S&P 500 headlines dosyası otomatik bulunamadı. "
            "Yukarıdaki dosya listesinden gerçek dosya adını kontrol et."
        )

    print("\nAday dosyalar:")
    for i, (score, p) in enumerate(candidate_files):
        print(i, "score:", score, "->", p.name)

    FILE_PATH = candidate_files[0][1]

print("\nSeçilen dosya:")
print(FILE_PATH)

# ------------------------------------------------------------
# 3) Output yolları
# ------------------------------------------------------------
OUT_PARQUET = paths.SP500_FINBERT_LABELED_HEADLINES_PARQUET_PATH
OUT_CSV = paths.SP500_FINBERT_LABELED_HEADLINES_PATH
OUT_XLSX = paths.SP500_FINBERT_LABELED_HEADLINES_XLSX_PATH
PROGRESS_PARQUET = paths.SP500_FINBERT_LABELING_PROGRESS_PATH

print("\nOutput files:")
print("PARQUET :", OUT_PARQUET)
print("CSV     :", OUT_CSV)
print("XLSX    :", OUT_XLSX)
print("PROGRESS:", PROGRESS_PARQUET)

# ------------------------------------------------------------
# 4) Dosyayı oku
# ------------------------------------------------------------
def read_any_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".parquet":
        return pd.read_parquet(path)

    if suffix == ".csv":
        encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
        last_err = None

        for enc in encodings:
            try:
                return pd.read_csv(path, encoding=enc)
            except Exception as e:
                last_err = e

        raise RuntimeError(f"CSV okunamadı. Son hata: {last_err}")

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if suffix == ".json":
        return pd.read_json(path)

    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)

    raise RuntimeError(f"Desteklenmeyen dosya uzantısı: {suffix}")

raw_df = read_any_file(FILE_PATH)

print("\nRaw shape:", raw_df.shape)
print("\nColumns:")
print(raw_df.columns.tolist())



# ------------------------------------------------------------
# 5) Text/headline kolonunu otomatik bul
# ------------------------------------------------------------
possible_text_cols = [
    "headline",
    "headlines",
    "title",
    "news_title",
    "news_headline",
    "text",
    "content",
    "summary",
    "article",
    "news",
]

lower_col_map = {c.lower().strip(): c for c in raw_df.columns}

text_col = None

for c in possible_text_cols:
    if c in lower_col_map:
        text_col = lower_col_map[c]
        break

if text_col is None:
    object_cols = raw_df.select_dtypes(include=["object", "string"]).columns.tolist()

    if not object_cols:
        raise ValueError("Text/headline kolonu bulunamadı. Kolonları kontrol et.")

    avg_lengths = {}
    for c in object_cols:
        avg_lengths[c] = raw_df[c].dropna().astype(str).str.len().mean()

    text_col = max(avg_lengths, key=avg_lengths.get)

print("\nSeçilen text kolonu:", text_col)

# ------------------------------------------------------------
# 6) Date kolonu varsa bul
# ------------------------------------------------------------
possible_date_cols = [
    "date",
    "datetime",
    "published_date",
    "publish_date",
    "time",
    "timestamp",
]

date_col = None

for c in possible_date_cols:
    if c in lower_col_map:
        date_col = lower_col_map[c]
        break

print("Seçilen date kolonu:", date_col)

# ------------------------------------------------------------
# 7) Temiz dataframe hazırla
# ------------------------------------------------------------
df = raw_df.copy()

df["text"] = df[text_col].astype(str).str.strip()

df = df[df["text"].notna()].copy()
df = df[df["text"] != ""].copy()
df = df[df["text"].str.lower() != "nan"].copy()

df["n_words"] = df["text"].str.split().str.len()
df["text_len"] = df["text"].str.len()

df = df[df["n_words"] >= 4].copy()
df = df[df["text_len"] >= 20].copy()

df["text_norm"] = (
    df["text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

before_dedup = len(df)
df = df.drop_duplicates(subset=["text_norm"]).copy()
after_dedup = len(df)

print("\nBefore dedup:", before_dedup)
print("After dedup :", after_dedup)

if date_col is not None:
    df["date"] = pd.to_datetime(df[date_col], errors="coerce")
else:
    df["date"] = pd.NaT

df = df.reset_index(drop=True)

if "sample_id" not in df.columns:
    df.insert(0, "sample_id", [f"SP500_HEAD_{i+1:06d}" for i in range(len(df))])

df["source_dataset"] = "SP500_Financial_News_Headlines_2008_2024"
df["source_file"] = FILE_PATH.name

# ------------------------------------------------------------
# 8) FinBERT kolonları - dtype güvenli
# ------------------------------------------------------------
# ÖNEMLİ:
# finbert_label object olmak zorunda.
# Yoksa "negative" yazarken float64 dtype hatası verir.

if "finbert_label" not in df.columns:
    df["finbert_label"] = pd.Series([pd.NA] * len(df), dtype="object")
else:
    df["finbert_label"] = df["finbert_label"].astype("object")

numeric_finbert_cols = [
    "finbert_confidence",
    "finbert_prob_negative",
    "finbert_prob_neutral",
    "finbert_prob_positive",
    "finbert_margin",
]

for col in numeric_finbert_cols:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\nClean shape:", df.shape)


# ------------------------------------------------------------
# 9) Progress varsa yükle
# ------------------------------------------------------------
if PROGRESS_PARQUET.exists():
    print("\nProgress dosyası bulundu, yükleniyor:")
    print(PROGRESS_PARQUET)

    progress_df = pd.read_parquet(PROGRESS_PARQUET)

    if len(progress_df) == len(df):
        df = progress_df.copy()

        # dtype güvenliği
        if "finbert_label" not in df.columns:
            df["finbert_label"] = pd.Series([pd.NA] * len(df), dtype="object")
        else:
            df["finbert_label"] = df["finbert_label"].astype("object")

        for col in numeric_finbert_cols:
            if col not in df.columns:
                df[col] = np.nan
            df[col] = pd.to_numeric(df[col], errors="coerce")

        print("Progress yüklendi. Shape:", df.shape)
        print("Etiketlenmiş satır:", df["finbert_label"].notna().sum())
    else:
        print("UYARI: Progress dosyasının satır sayısı mevcut df ile aynı değil. Progress kullanılmadı.")
else:
    print("\nProgress dosyası yok. Baştan başlanacak.")

# ------------------------------------------------------------
# 10) FinBERT modeli yükle
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

print("\nFinBERT id2label:")
print(model.config.id2label)

def normalize_finbert_label(label):
    s = str(label).lower().strip()

    if "negative" in s:
        return "negative"
    if "neutral" in s:
        return "neutral"
    if "positive" in s:
        return "positive"

    mapping = {
        "label_0": "positive",
        "label_1": "negative",
        "label_2": "neutral",
    }

    return mapping.get(s, s)

id_to_norm_label = {
    int(i): normalize_finbert_label(lbl)
    for i, lbl in model.config.id2label.items()
}

print("\nNormalized id2label:")
print(id_to_norm_label)

# ------------------------------------------------------------
# 11) Eksik kalanları FinBERT ile label'la
# ------------------------------------------------------------
missing_mask = (
    df["finbert_label"].isna() |
    df["finbert_label"].astype(str).str.strip().str.lower().isin(["", "nan", "none", "<na>"])
)

missing_indices = df.index[missing_mask].tolist()

print("\nToplam satır:", len(df))
print("Label eksik satır:", len(missing_indices))
print("Label hazır satır:", len(df) - len(missing_indices))

@torch.no_grad()
def predict_batch(batch_texts):
    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    enc = {k: v.to(device) for k, v in enc.items()}

    outputs = model(**enc)
    probs = torch.softmax(outputs.logits, dim=-1).detach().cpu().numpy()
    pred_ids = probs.argmax(axis=1)

    rows = []

    for pred_id, prob_vec in zip(pred_ids, probs):
        label = id_to_norm_label[int(pred_id)]
        conf = float(prob_vec[int(pred_id)])

        prob_map = {
            "negative": np.nan,
            "neutral": np.nan,
            "positive": np.nan,
        }

        for class_id, p in enumerate(prob_vec):
            norm_label = id_to_norm_label[int(class_id)]
            if norm_label in prob_map:
                prob_map[norm_label] = float(p)

        prob_values = [
            prob_map["negative"],
            prob_map["neutral"],
            prob_map["positive"],
        ]

        sorted_probs = sorted(
            [p for p in prob_values if not pd.isna(p)],
            reverse=True
        )

        if len(sorted_probs) >= 2:
            margin = float(sorted_probs[0] - sorted_probs[1])
        else:
            margin = np.nan

        rows.append({
            "finbert_label": label,
            "finbert_confidence": conf,
            "finbert_prob_negative": prob_map["negative"],
            "finbert_prob_neutral": prob_map["neutral"],
            "finbert_prob_positive": prob_map["positive"],
            "finbert_margin": margin,
        })

    return rows

num_batches = int(np.ceil(len(missing_indices) / BATCH_SIZE))

for batch_no in tqdm(range(num_batches), desc="FinBERT labeling missing rows"):
    start = batch_no * BATCH_SIZE
    end = start + BATCH_SIZE

    batch_indices = missing_indices[start:end]
    batch_texts = df.loc[batch_indices, "text"].astype(str).tolist()

    preds = predict_batch(batch_texts)

    for idx, pred in zip(batch_indices, preds):
        # label string/object
        df.at[idx, "finbert_label"] = str(pred["finbert_label"])

        # numeric columns
        df.at[idx, "finbert_confidence"] = float(pred["finbert_confidence"])
        df.at[idx, "finbert_prob_negative"] = float(pred["finbert_prob_negative"])
        df.at[idx, "finbert_prob_neutral"] = float(pred["finbert_prob_neutral"])
        df.at[idx, "finbert_prob_positive"] = float(pred["finbert_prob_positive"])
        df.at[idx, "finbert_margin"] = float(pred["finbert_margin"])

    # Ara kayıt
    if (batch_no + 1) % SAVE_EVERY_BATCHES == 0:
        df.to_parquet(PROGRESS_PARQUET, index=False)
        print(f"\nProgress kaydedildi: batch {batch_no + 1}/{num_batches}")
        print("Etiketlenmiş satır:", df["finbert_label"].notna().sum())

# Son progress kaydı
df.to_parquet(PROGRESS_PARQUET, index=False)

print("\nFinBERT labeling tamamlandı.")
print("Etiketlenmiş satır:", df["finbert_label"].notna().sum())

# ------------------------------------------------------------
# 12) Özet
# ------------------------------------------------------------
print("\nFinBERT label distribution:")
display(pd.DataFrame({
    "count": df["finbert_label"].value_counts(dropna=False),
    "ratio_%": df["finbert_label"].value_counts(dropna=False, normalize=True).mul(100).round(2)
}))

print("\nFinBERT confidence summary:")
display(df["finbert_confidence"].describe())

print("\nFinBERT margin summary:")
display(df["finbert_margin"].describe())


# ------------------------------------------------------------
# 13) Kaydet: _labeled
# ------------------------------------------------------------
df.to_parquet(OUT_PARQUET, index=False)
df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")

# Excel sınırı 1,048,576 satır.
if len(df) <= 1_048_576:
    df.to_excel(OUT_XLSX, index=False)
else:
    print("Excel satır sınırı nedeniyle xlsx kaydedilmedi.")

print("\n" + "=" * 100)
print("KAYIT TAMAMLANDI")
print("=" * 100)
print("Parquet :", OUT_PARQUET)
print("CSV     :", OUT_CSV)

if len(df) <= 1_048_576:
    print("Excel   :", OUT_XLSX)

print("Progress:", PROGRESS_PARQUET)

print("\nOrijinal dosya bozulmadı:")
print(FILE_PATH)

In [ ]:
# ============================================================
# ANNOTATION İÇİN TEMİZ VE ORTA-UZUN HEADLINE SETİ
# ============================================================

clean_df = df.copy()

clean_df["text"] = clean_df["text"].astype(str).str.strip()
clean_df["n_words"] = clean_df["text"].str.split().str.len()
clean_df["text_len"] = clean_df["text"].str.len()

clean_df = clean_df[
    (clean_df["text"].notna()) &
    (clean_df["text"] != "") &
    (clean_df["text"].str.lower() != "nan") &
    (clean_df["text"].str.strip() != "...") &
    (clean_df["n_words"] >= 6) &
    (clean_df["text_len"] >= 40) &
    (clean_df["text_len"] <= 180)
].copy()

clean_df["text_norm"] = (
    clean_df["text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

clean_df = clean_df.drop_duplicates(subset=["text_norm"]).reset_index(drop=True)

print("Orijinal df:", df.shape)
print("Temiz clean_df:", clean_df.shape)

print("\nFinBERT label dağılımı:")
print(clean_df["finbert_label"].value_counts(dropna=False))

print("\nUzunluk özeti:")
display(clean_df[["n_words", "text_len"]].describe())

print("\nİlk 3 temiz örnek:")
for i, row in clean_df.head(PREVIEW_ROWS).iterrows():
    print("=" * 100)
    print("sample_id:", row.get("sample_id", ""))
    print("date:", row.get("date", ""))
    print("n_words:", row["n_words"], "| text_len:", row["text_len"])
    print("finbert_label:", row.get("finbert_label", ""))
    print("finbert_confidence:", row.get("finbert_confidence", ""))
    print("TEXT:")
    print(row["text"])

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import math

# ============================================================
# clean_df -> DENGELİ 1500 ANNOTATION SET
# 10'ARLIK BATCH DOSYALARI
#
# Mantık:
# - clean_df içinden seçer
# - FinBERT label'a göre dengeli seçer:
#   negative: 500
#   neutral : 500
#   positive: 500
# - 10'ar satırlık xlsx/csv/txt batch dosyaları oluşturur
# ============================================================

DATA_DIR = DATA_ROOT

OUT_DIR = paths.SP500_ANNOTATION_PREPARATION_BATCHES_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
N_TOTAL = 1500
BATCH_SIZE = 10

if "clean_df" not in globals():
    raise ValueError("clean_df bulunamadı. Önce clean_df oluşturma hücresini çalıştır.")

df = clean_df.copy()

# ------------------------------------------------------------
# 1) Temel kontroller
# ------------------------------------------------------------
required_cols = ["text", "finbert_label", "finbert_confidence"]

for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"clean_df içinde eksik kolon: {col}")

df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""].copy()
df = df[df["text"].str.lower() != "nan"].copy()
df = df[df["text"].str.strip() != "..."].copy()

if "text_norm" not in df.columns:
    df["text_norm"] = (
        df["text"]
        .str.lower()
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

df = df.drop_duplicates(subset=["text_norm"]).reset_index(drop=True)

# Label normalize
df["finbert_label"] = (
    df["finbert_label"]
    .astype(str)
    .str.lower()
    .str.strip()
)

valid_labels = ["negative", "neutral", "positive"]
df = df[df["finbert_label"].isin(valid_labels)].copy().reset_index(drop=True)

print("Clean df shape:", df.shape)
print("\nFinBERT label distribution:")
print(df["finbert_label"].value_counts(dropna=False))

print("\nFinBERT label ratio:")
print(df["finbert_label"].value_counts(normalize=True, dropna=False).mul(100).round(2))


# ------------------------------------------------------------
# 2) 1500 örneği FinBERT label'a göre dengeli seç
# ------------------------------------------------------------
target_counts = {
    "negative": 500,
    "neutral": 500,
    "positive": 500,
}

parts = []
missing_total = 0

for label, n in target_counts.items():
    temp = df[df["finbert_label"] == label].copy()
    
    if len(temp) < n:
        print(f"UYARI: {label} için yeterli örnek yok. İstenen: {n}, mevcut: {len(temp)}")
        sample_n = len(temp)
        missing_total += n - len(temp)
    else:
        sample_n = n
    
    sampled = temp.sample(n=sample_n, random_state=RANDOM_STATE)
    parts.append(sampled)

annotation_df = pd.concat(parts, ignore_index=True)

# Eğer bir sınıfta eksik kaldıysa, kalan örneklerden tamamla
if len(annotation_df) < N_TOTAL:
    used = set(annotation_df["text_norm"])
    remaining = df[~df["text_norm"].isin(used)].copy()
    need = N_TOTAL - len(annotation_df)
    
    print(f"\nEksik kalan örnek sayısı: {need}")
    print("Kalan havuz:", remaining.shape)
    
    if len(remaining) > 0:
        add_n = min(need, len(remaining))
        annotation_df = pd.concat(
            [
                annotation_df,
                remaining.sample(n=add_n, random_state=RANDOM_STATE)
            ],
            ignore_index=True
        )

# Eğer herhangi bir sebeple fazla olduysa 1500'e indir
if len(annotation_df) > N_TOTAL:
    annotation_df = annotation_df.sample(n=N_TOTAL, random_state=RANDOM_STATE)

# Karıştır
annotation_df = annotation_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("\nSeçilen annotation df shape:", annotation_df.shape)
print("\nSeçilen annotation FinBERT label distribution:")
print(annotation_df["finbert_label"].value_counts(dropna=False))

print("\nSeçilen annotation FinBERT label ratio:")
print(annotation_df["finbert_label"].value_counts(normalize=True, dropna=False).mul(100).round(2))


# ------------------------------------------------------------
# 3) annotation_id oluştur
# ------------------------------------------------------------
if "annotation_id" in annotation_df.columns:
    annotation_df = annotation_df.drop(columns=["annotation_id"])

annotation_df.insert(
    0,
    "annotation_id",
    [f"SP500_ANN_{i+1:04d}" for i in range(len(annotation_df))]
)

# Sonradan doldurulacak kolonlar
annotation_df["chatgpt_label"] = ""
annotation_df["chatgpt_confidence"] = ""
annotation_df["chatgpt_reason"] = ""
annotation_df["final_label"] = ""
annotation_df["annotation_note"] = ""

# ------------------------------------------------------------
# 4) Kolon sırası
# ------------------------------------------------------------
keep_cols = [
    "annotation_id",
    "sample_id",
    "date",
    "text",
    "finbert_label",
    "finbert_confidence",
    "finbert_prob_negative",
    "finbert_prob_neutral",
    "finbert_prob_positive",
    "finbert_margin",
    "chatgpt_label",
    "chatgpt_confidence",
    "chatgpt_reason",
    "final_label",
    "annotation_note",
    "n_words",
    "text_len",
]

annotation_df = annotation_df[[c for c in keep_cols if c in annotation_df.columns]].copy()

print("\nFinal annotation df shape:", annotation_df.shape)
display(annotation_df.head(PREVIEW_ROWS))


# ------------------------------------------------------------
# 5) Ana annotation dosyasını kaydet
# ------------------------------------------------------------
MAIN_XLSX = paths.SP500_ANNOTATION_MASTER_XLSX_PATH
MAIN_CSV = paths.SP500_ANNOTATION_MASTER_CSV_PATH
MAIN_PARQUET = paths.SP500_ANNOTATION_MASTER_PARQUET_PATH

annotation_df.to_excel(MAIN_XLSX, index=False)
annotation_df.to_csv(MAIN_CSV, index=False, encoding="utf-8-sig")
annotation_df.to_parquet(MAIN_PARQUET, index=False)

print("\nMaster annotation dosyaları kaydedildi:")
print(MAIN_XLSX)
print(MAIN_CSV)
print(MAIN_PARQUET)


# ------------------------------------------------------------
# 6) 10'arlı batch dosyaları oluştur
# ------------------------------------------------------------
n_batches = math.ceil(len(annotation_df) / BATCH_SIZE)

for batch_no in range(n_batches):
    start = batch_no * BATCH_SIZE
    end = start + BATCH_SIZE
    
    batch_df = annotation_df.iloc[start:end].copy()
    
    batch_id = batch_no + 1
    
    batch_xlsx = OUT_DIR / f"annotation_batch_{batch_id:03d}.xlsx"
    batch_csv = OUT_DIR / f"annotation_batch_{batch_id:03d}.csv"
    batch_txt = OUT_DIR / f"annotation_batch_{batch_id:03d}.txt"
    
    batch_df.to_excel(batch_xlsx, index=False)
    batch_df.to_csv(batch_csv, index=False, encoding="utf-8-sig")
    
    # Bana atman için tam metinli TXT
    with open(batch_txt, "w", encoding="utf-8") as f:
        f.write("=" * 120 + "\n")
        f.write(f"ANNOTATION BATCH {batch_id:03d}\n")
        f.write(f"Rows: {start} - {start + len(batch_df) - 1}\n")
        f.write("=" * 120 + "\n\n")
        
        for _, row in batch_df.iterrows():
            f.write("-" * 120 + "\n")
            f.write(f"annotation_id: {row.get('annotation_id', '')}\n")
            f.write(f"sample_id: {row.get('sample_id', '')}\n")
            f.write(f"date: {row.get('date', '')}\n")
            f.write(f"finbert_label: {row.get('finbert_label', '')}\n")
            f.write(f"finbert_confidence: {row.get('finbert_confidence', '')}\n")
            f.write("TEXT:\n")
            f.write(str(row.get("text", "")) + "\n\n")

print("\nBatch dosyaları oluşturuldu.")
print("Batch sayısı:", n_batches)
print("Batch klasörü:", OUT_DIR)


# ------------------------------------------------------------
# 7) İstediğin batch'i notebookta tam metin yazdırma fonksiyonu
# ------------------------------------------------------------
def print_annotation_batch(batch_no=1, preview_rows=PREVIEW_ROWS):
    start = (batch_no - 1) * BATCH_SIZE
    end = start + BATCH_SIZE
    
    temp = annotation_df.iloc[start:end].head(preview_rows).copy()
    
    print("=" * 120)
    print(f"ANNOTATION BATCH {batch_no:03d}")
    print(f"Rows: {start} - {start + len(temp) - 1}")
    print("=" * 120)
    
    for _, row in temp.iterrows():
        print("-" * 120)
        print("annotation_id:", row.get("annotation_id", ""))
        print("sample_id:", row.get("sample_id", ""))
        print("date:", row.get("date", ""))
        print("finbert_label:", row.get("finbert_label", ""))
        print("finbert_confidence:", row.get("finbert_confidence", ""))
        print("TEXT:")
        print(row.get("text", ""))
        print()


In [ ]:

# İlk batch'i yazdır
print_annotation_batch(batch_no=3)